# Lab – Step 4: KB Expansion via SPARQL

**Target size:** 50,000–200,000 triples · 5,000–30,000 entities · 50–200 relations

**Expansion strategy (anchored):**
1. **1-Hop** — for each confidently aligned entity, pull all its Wikidata triples
2. **Predicate-controlled** — pull all instances of our key aligned predicates
3. **2-Hop** — follow specific high-value relation chains (award → winners, genre → artists, etc.)
4. **Clean** — deduplicate, filter predicates, ensure connectivity
5. **Export** — TSV format ready for KGE frameworks (PyKEEN, AmpliGraph, etc.)

## 0. Install & import

In [1]:
!pip install rdflib requests pandas ipywidgets SPARQLWrapper tqdm --quiet

In [2]:
import time
import json
import hashlib
import pandas as pd
import collections
from tqdm.notebook import tqdm
from SPARQLWrapper import SPARQLWrapper, JSON
from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS, OWL, XSD

MUS   = Namespace("http://example.org/music/")
PROP  = Namespace("http://example.org/music/prop/")
WD    = Namespace("http://www.wikidata.org/entity/")
WDT   = Namespace("http://www.wikidata.org/prop/direct/")
SKOS  = Namespace("http://www.w3.org/2004/02/skos/core#")
SCHEMA= Namespace("https://schema.org/")

# Load aligned KB from Step 3
g = Graph()
g.parse("music_kb_aligned.ttl", format="turtle")
for prefix, ns in [("mus",MUS),("prop",PROP),("wd",WD),("wdt",WDT),
                   ("owl",OWL),("rdfs",RDFS),("skos",SKOS),("xsd",XSD)]:
    g.bind(prefix, ns)

# Wikidata SPARQL endpoint
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)
sparql.addCustomHttpHeader("User-Agent", "MusicKB-Lab/4.0 (expansion)")

print(f"KB loaded: {len(g)} triples")

KB loaded: 876 triples


## 1. Collect aligned entity QIDs from our KB

In [3]:
# Extract all owl:sameAs links that point to Wikidata
aligned = {}  # local_uri -> wikidata_qid
for s, p, o in g.triples((None, OWL.sameAs, None)):
    o_str = str(o)
    if "wikidata.org/entity/Q" in o_str:
        qid = o_str.split("/")[-1]
        # only keep entities from our MUS namespace
        if str(s).startswith(str(MUS)):
            cls = g.value(s, RDF.type)
            cls_name = str(cls).split("/")[-1] if cls else "Unknown"
            aligned[str(s)] = (qid, cls_name)

print(f"Aligned entities: {len(aligned)}")
by_class = collections.Counter(v[1] for v in aligned.values())
for cls, cnt in sorted(by_class.items()):
    print(f"  {cls:<15} {cnt}")

Aligned entities: 64
  Album           15
  Artist          21
  Award           8
  Country         5
  Genre           9
  RecordLabel     6


## 2. SPARQL helpers with retry & rate-limiting

In [4]:
# Predicates to EXCLUDE from expansion (too noisy / literal-heavy for KGE)
BLOCKED_PREDICATES = {
    # Wikidata internal / administrative
    "P18",   # image
    "P94",   # coat of arms image
    "P154",  # logo image
    "P856",  # official website
    "P18",   # image
    "P373",  # Commons category
    "P935",  # Commons gallery
    "P948",  # page banner
    "P910",  # topic's main category
    "P1566", # GeoNames ID
    "P1667", # Getty TGN ID
    "P2163", # FAST ID
    "P244",  # Library of Congress auth ID
    "P268",  # BnF ID
    "P269",  # SUDOC auth ID
    "P227",  # GND ID
    "P345",  # IMDb ID
    "P646",  # Freebase ID
    "P1143", # BN (bibliographic) ID
    "P3417", # Quora topic ID
    "P4839", # Wolfram Language entity
    "P5008", # on focus list of Wikimedia project
    "P2397", # YouTube channel ID
    "P2002", # Twitter username
    "P2003", # Instagram username
    "P4013", # Giphy username
}

BLOCK_FILTER = " ".join(
    f"FILTER(?p != wdt:{pid})" for pid in BLOCKED_PREDICATES
)

def run_sparql(query: str, retries: int = 4, delay: float = 1.5) -> list:
    for attempt in range(retries):
        try:
            sparql.setQuery(query)
            data = sparql.query().convert()
            vars_ = data["head"]["vars"]
            return [{v: b[v]["value"] if v in b else "" for v in vars_}
                    for b in data["results"]["bindings"]]
        except Exception as e:
            wait = delay * (2 ** attempt)
            print(f"  [WARN] attempt {attempt+1} failed ({e}), retrying in {wait:.0f}s")
            time.sleep(wait)
    return []


def add_wikidata_triple(s_str, p_str, o_str):
    """Convert raw Wikidata URIs/literals to RDFLib terms and add to graph."""
    # Subject
    s = URIRef(s_str)
    # Predicate — keep only wdt: direct claims
    if not p_str.startswith("http://www.wikidata.org/prop/direct/"):
        return False
    pid = p_str.split("/")[-1]
    if pid in BLOCKED_PREDICATES:
        return False
    p = URIRef(p_str)
    # Object — URI or Literal
    if o_str.startswith("http"):
        o = URIRef(o_str)
    else:
        o = Literal(o_str)
    g.add((s, p, o))
    return True

print("Helpers ready.")

Helpers ready.


## 3. Wave 1 — 1-Hop expansion from every aligned entity

```sparql
SELECT ?p ?o WHERE { wd:QXXX ?p ?o . } LIMIT 1000
```

In [5]:
WAVE1_LIMIT = 300   # triples per entity (stay polite)
wave1_added = 0

print(f"Wave 1: 1-hop from {len(aligned)} aligned entities ...")

for local_uri, (qid, cls_name) in tqdm(aligned.items()):
    q = f"""
SELECT ?p ?o WHERE {{
  wd:{qid} ?p ?o .
  FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
}}
LIMIT {WAVE1_LIMIT}
"""
    rows = run_sparql(q)
    for r in rows:
        if add_wikidata_triple(
                f"http://www.wikidata.org/entity/{qid}",
                r["p"], r["o"]):
            wave1_added += 1
    time.sleep(0.8)

print(f"Wave 1 done. Added {wave1_added} triples. Total: {len(g)}")

Wave 1: 1-hop from 64 aligned entities ...


  0%|          | 0/64 [00:00<?, ?it/s]

Wave 1 done. Added 5856 triples. Total: 6722


## 4. Wave 2 — Predicate-controlled expansion

Pull all subjects/objects connected via our most important aligned predicates.

```sparql
SELECT ?s ?o WHERE { ?s wdt:P166 ?o . } LIMIT 20000
```

In [6]:
# Key predicate–controlled expansions (pid, limit, description)
PREDICATE_EXPANSIONS = [
    ("P136", 8000,  "genre — music entities with genre links"),
    ("P166", 8000,  "award received — entities that won music awards"),
    ("P264", 6000,  "record label — artists signed to labels"),
    ("P737", 4000,  "influenced by — influence chains"),
    ("P175", 6000,  "performer — works and their performers"),
    ("P495", 4000,  "country of origin — music from specific countries"),
    ("P279", 3000,  "subclass of — genre taxonomy"),
    ("P577", 5000,  "publication date — albums/songs"),
    ("P571", 3000,  "inception — label founding"),
    ("P569", 4000,  "date of birth — musicians"),
    ("P2031",3000,  "work period start — career starts"),
]

# Restrict to music-domain by requiring subject to be instance of musical entity
MUSIC_TYPES = " ".join([
    "wd:Q5",       # human
    "wd:Q215380",  # band
    "wd:Q134556",  # single
    "wd:Q482994",  # album
    "wd:Q105543609",# musical work
    "wd:Q18127",   # record label
    "wd:Q188451",  # music genre
])

wave2_added = 0

for pid, lim, desc in tqdm(PREDICATE_EXPANSIONS):
    q = f"""
SELECT ?s ?o WHERE {{
  ?s wdt:{pid} ?o .
  ?s wdt:P31 ?type .
  VALUES ?type {{ {MUSIC_TYPES} }}
}}
LIMIT {lim}
"""
    rows = run_sparql(q)
    p_uri = f"http://www.wikidata.org/prop/direct/{pid}"
    for r in rows:
        if add_wikidata_triple(r["s"], p_uri, r["o"]):
            wave2_added += 1
    print(f"  P{pid} ({desc}): {len(rows)} rows fetched")
    time.sleep(1.2)

print(f"\nWave 2 done. Added {wave2_added} triples. Total: {len(g)}")

  0%|          | 0/11 [00:00<?, ?it/s]

  PP136 (genre — music entities with genre links): 8000 rows fetched
  PP166 (award received — entities that won music awards): 8000 rows fetched
  PP264 (record label — artists signed to labels): 6000 rows fetched
  PP737 (influenced by — influence chains): 4000 rows fetched
  PP175 (performer — works and their performers): 6000 rows fetched
  PP495 (country of origin — music from specific countries): 4000 rows fetched
  [WARN] attempt 1 failed (HTTP Error 504: Gateway Timeout), retrying in 2s
  [WARN] attempt 2 failed (HTTP Error 504: Gateway Timeout), retrying in 3s
  [WARN] attempt 3 failed (HTTP Error 504: Gateway Timeout), retrying in 6s
  [WARN] attempt 4 failed (HTTP Error 504: Gateway Timeout), retrying in 12s
  PP279 (subclass of — genre taxonomy): 0 rows fetched
  PP577 (publication date — albums/songs): 5000 rows fetched
  PP571 (inception — label founding): 3000 rows fetched
  PP569 (date of birth — musicians): 4000 rows fetched
  PP2031 (work period start — career starts)

## 5. Wave 3 — 2-Hop expansion

Follow high-value chains to bring in neighbouring entities.

```sparql
SELECT ?award ?p ?o WHERE {
  ?person wdt:P166 ?award .
  ?award ?p ?o .
} LIMIT 10000
```

In [7]:
TWO_HOP_QUERIES = [
    (
        "2-hop: award neighbours",
        """
SELECT ?award ?p ?o WHERE {
  ?person wdt:P166 ?award .
  ?person wdt:P31 wd:Q5 .
  ?award ?p ?o .
  FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
}
LIMIT 10000
""",
    ),
    (
        "2-hop: genre taxonomy",
        """
SELECT ?subgenre ?p ?o WHERE {
  ?artist wdt:P136 ?subgenre .
  ?artist wdt:P31 wd:Q5 .
  ?subgenre ?p ?o .
  FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
}
LIMIT 8000
""",
    ),
    (
        "2-hop: record label details",
        """
SELECT ?label ?p ?o WHERE {
  ?artist wdt:P264 ?label .
  ?artist wdt:P31 wd:Q5 .
  ?label ?p ?o .
  FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
}
LIMIT 8000
""",
    ),
    (
        "2-hop: influence network",
        """
SELECT ?influencer ?p ?o WHERE {
  ?artist wdt:P737 ?influencer .
  ?artist wdt:P31 wd:Q5 .
  ?influencer ?p ?o .
  FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
}
LIMIT 8000
""",
    ),
    (
        "2-hop: album details from performers",
        """
SELECT ?album ?p ?o WHERE {
  ?album wdt:P175 ?artist .
  ?album wdt:P31 wd:Q482994 .
  ?album ?p ?o .
  FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
}
LIMIT 10000
""",
    ),
    (
        "2-hop: country -> music artists",
        """
SELECT ?artist ?p ?o WHERE {
  ?artist wdt:P495 ?country .
  ?country wdt:P31 wd:Q6256 .
  ?artist ?p ?o .
  FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
}
LIMIT 10000
""",
    ),
]

wave3_added = 0
for desc, q in tqdm(TWO_HOP_QUERIES):
    rows = run_sparql(q)
    for r in rows:
        # variable names differ per query — take first three bound values
        vals = [v for v in r.values() if v]
        if len(vals) >= 3:
            if add_wikidata_triple(vals[0], vals[1], vals[2]):
                wave3_added += 1
    print(f"  {desc}: {len(rows)} rows, total graph: {len(g)}")
    time.sleep(1.5)

print(f"\nWave 3 done. Added {wave3_added} triples. Total: {len(g)}")

  0%|          | 0/6 [00:00<?, ?it/s]

  2-hop: award neighbours: 10000 rows, total graph: 60373
  2-hop: genre taxonomy: 8000 rows, total graph: 61375
  2-hop: record label details: 8000 rows, total graph: 63077
  2-hop: influence network: 8000 rows, total graph: 65010
  2-hop: album details from performers: 10000 rows, total graph: 73745
  2-hop: country -> music artists: 10000 rows, total graph: 83330

Wave 3 done. Added 49198 triples. Total: 83330


## 6. Wave 4 — Top-up if still below 50,000 triples

Broad music-domain sweep to ensure minimum size requirement.

In [8]:
TARGET_MIN = 50_000
TOPUP_BATCH = 15_000

topup_queries = [
    ("musicians born after 1960",
     f"""
SELECT ?s ?p ?o WHERE {{
  ?s wdt:P31 wd:Q5 .
  ?s wdt:P106 wd:Q639669 .
  ?s ?p ?o .
  FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
  FILTER(?p != wdt:P18)
}}
LIMIT {TOPUP_BATCH}
"""),
    ("studio albums",
     f"""
SELECT ?s ?p ?o WHERE {{
  ?s wdt:P31 wd:Q208569 .
  ?s ?p ?o .
  FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
}}
LIMIT {TOPUP_BATCH}
"""),
    ("music genres taxonomy",
     f"""
SELECT ?s ?p ?o WHERE {{
  ?s wdt:P31 wd:Q188451 .
  ?s ?p ?o .
  FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
}}
LIMIT {TOPUP_BATCH}
"""),
    ("record labels",
     f"""
SELECT ?s ?p ?o WHERE {{
  ?s wdt:P31 wd:Q18127 .
  ?s ?p ?o .
  FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
}}
LIMIT {TOPUP_BATCH}
"""),
]

topup_added = 0
for desc, q in topup_queries:
    if len(g) >= TARGET_MIN:
        print(f"Target {TARGET_MIN:,} reached — stopping top-up.")
        break
    rows = run_sparql(q)
    for r in rows:
        vals = list(r.values())
        if len(vals) >= 3 and all(vals[:3]):
            if add_wikidata_triple(vals[0], vals[1], vals[2]):
                topup_added += 1
    print(f"  Top-up '{desc}': {len(rows)} rows, total: {len(g):,}")
    time.sleep(2.0)

print(f"\nWave 4 done. Added {topup_added} triples. Total: {len(g):,}")

Target 50,000 reached — stopping top-up.

Wave 4 done. Added 0 triples. Total: 83,330


## 7. Cleaning pipeline

**Steps:**
1. Remove pure-literal triples (strings, numbers with no semantic value for KGE)
2. Remove isolated nodes (no connections)
3. Normalise predicates (keep only `wdt:` and `prop:` namespaces)
4. Count and report

In [9]:
from rdflib import Literal

print(f"Before cleaning: {len(g):,} triples")

# ── Step 7a: Remove triples with Literal objects ─────────────────────────
# For KGE we only want (entity, relation, entity) triples.
# KEEP literals that are typed dates or numbers (may be used as features),
# but REMOVE long string literals (descriptions, labels for non-seed entities)
KEEP_LITERAL_DATATYPES = {XSD.gYear, XSD.date, XSD.dateTime,
                          XSD.decimal, XSD.integer, XSD.float}

to_remove = []
for s, p, o in g:
    if isinstance(o, Literal):
        # Keep typed numerics / dates from our PROP namespace
        if str(p).startswith(str(PROP)) and o.datatype in KEEP_LITERAL_DATATYPES:
            continue
        # Remove all other literals from Wikidata (labels, descriptions, etc.)
        to_remove.append((s, p, o))

for triple in to_remove:
    g.remove(triple)
print(f"After removing unwanted literals: {len(g):,} triples")

# ── Step 7b: Remove administrative / provenance triples ──────────────────
REMOVE_PRED_PREFIXES = [
    "http://www.wikidata.org/prop/qualifier/",
    "http://www.wikidata.org/prop/reference/",
    "http://www.wikidata.org/prop/statement/",
    "http://schema.org/dateModified",
    "http://schema.org/version",
]
to_remove = [(s,p,o) for s,p,o in g
             if any(str(p).startswith(pfx) for pfx in REMOVE_PRED_PREFIXES)]
for triple in to_remove:
    g.remove(triple)
print(f"After removing admin predicates:  {len(g):,} triples")

# ── Step 7c: Keep only URI->URI triples for the final KGE export ─────────
# (we save a separate version; the full graph keeps typed literals)
kge_triples = [(s, p, o) for s, p, o in g
               if isinstance(s, URIRef) and isinstance(o, URIRef)]
print(f"URI->URI triples (KGE-ready):     {len(kge_triples):,}")

Before cleaning: 83,330 triples
After removing unwanted literals: 52,037 triples
After removing admin predicates:  52,037 triples
URI->URI triples (KGE-ready):     51,961


## 8. Connectivity analysis — remove isolated nodes

In [10]:
from collections import defaultdict

# Build adjacency for URI->URI triples
degree = defaultdict(int)
for s, p, o in kge_triples:
    degree[str(s)] += 1
    degree[str(o)] += 1

isolated = {uri for uri, deg in degree.items() if deg < 2}
print(f"Entities with degree < 2 (isolated): {len(isolated):,}")

# Remove triples where BOTH s and o are isolated
kge_triples = [(s, p, o) for s, p, o in kge_triples
               if not (str(s) in isolated and str(o) in isolated)]
print(f"KGE triples after isolation removal: {len(kge_triples):,}")

# Degree distribution
degrees = sorted(degree.values(), reverse=True)
print(f"\nNode degree stats:")
print(f"  Max degree  : {degrees[0]}")
print(f"  Median      : {degrees[len(degrees)//2]}")
print(f"  Nodes deg>=5: {sum(1 for d in degrees if d >= 5)}")

Entities with degree < 2 (isolated): 37,807
KGE triples after isolation removal: 51,806

Node degree stats:
  Max degree  : 5404
  Median      : 1
  Nodes deg>=5: 1762


## 9. Final statistics

In [11]:
# Build KGE graph
g_kge = Graph()
for triple in kge_triples:
    g_kge.add(triple)

entities_kge  = set()
relations_kge = set()
for s, p, o in g_kge:
    entities_kge.add(str(s))
    entities_kge.add(str(o))
    relations_kge.add(str(p))

n_triples   = len(g_kge)
n_entities  = len(entities_kge)
n_relations = len(relations_kge)

print("=" * 55)
print(f"  KGE triples   : {n_triples:>10,}")
print(f"  Entities      : {n_entities:>10,}")
print(f"  Relations     : {n_relations:>10,}")
print("=" * 55)

ok_t = 50_000  <= n_triples   <= 200_000
ok_e = 5_000   <= n_entities  <= 30_000
ok_r = 50      <= n_relations <= 200

print(f"  Triples target   [50K–200K]  : {'PASS' if ok_t else 'FAIL'}")
print(f"  Entities target  [5K–30K]    : {'PASS' if ok_e else 'FAIL'}")
print(f"  Relations target [50–200]    : {'PASS' if ok_r else 'FAIL'}")

if not ok_t:
    if n_triples < 50_000:
        print("\n  [ACTION] Too few triples — re-run Wave 4 with larger TOPUP_BATCH.")
    else:
        print("\n  [ACTION] Too many triples — increase literal filtering or reduce LIMIT values.")
if not ok_r:
    print("\n  [ACTION] Relation count out of range — see Section 9b below.")

  KGE triples   :     51,806
  Entities      :     42,032
  Relations     :        439
  Triples target   [50K–200K]  : PASS
  Entities target  [5K–30K]    : FAIL
  Relations target [50–200]    : FAIL

  [ACTION] Relation count out of range — see Section 9b below.


In [12]:
# 9b. Relation distribution (top 30)
rel_counts = collections.Counter()
for s, p, o in g_kge:
    rel_counts[str(p).split("/")[-1]] += 1

df_rels = pd.DataFrame(rel_counts.most_common(40),
                       columns=["Predicate", "Count"])
print("Top 40 relations in KGE graph:")
display(df_rels)

Top 40 relations in KGE graph:


,Predicate,Count
0,P136,8943
1,P166,8205
2,P264,6557
3,P175,6403
4,P495,4622
5,P737,3964
6,P31,1174
7,P530,597
8,P658,548
9,P437,476


## 10. Export for KGE

**Three export formats:**
- `music_kg_kge.ttl` — full Turtle (for SPARQL / RDFLib downstream)
- `music_kg_triples.tsv` — tab-separated `subject\tpredicate\tobject` (PyKEEN / AmpliGraph)
- `music_kg_entities.tsv` and `music_kg_relations.tsv` — entity/relation dictionaries

In [13]:
# ── 10a. Turtle ───────────────────────────────────────────────────────────
g_kge.serialize(destination="music_kg_kge.ttl", format="turtle")
print("Saved -> music_kg_kge.ttl")

# ── 10b. TSV triples ──────────────────────────────────────────────────────
rows_tsv = [(str(s), str(p), str(o)) for s, p, o in g_kge]
df_triples = pd.DataFrame(rows_tsv, columns=["subject", "predicate", "object"])
df_triples.drop_duplicates(inplace=True)
df_triples.to_csv("music_kg_triples.tsv", sep="\t", index=False)
print(f"Saved -> music_kg_triples.tsv  ({len(df_triples):,} rows)")

# ── 10c. Entity dictionary ────────────────────────────────────────────────
ent_list = sorted(entities_kge)
df_ent = pd.DataFrame({"id": range(len(ent_list)), "entity": ent_list})
df_ent.to_csv("music_kg_entities.tsv", sep="\t", index=False)
print(f"Saved -> music_kg_entities.tsv  ({len(df_ent):,} entities)")

# ── 10d. Relation dictionary ──────────────────────────────────────────────
rel_list = sorted(relations_kge)
df_rel = pd.DataFrame({"id": range(len(rel_list)), "relation": rel_list})
df_rel.to_csv("music_kg_relations.tsv", sep="\t", index=False)
print(f"Saved -> music_kg_relations.tsv ({len(df_rel):,} relations)")

Saved -> music_kg_kge.ttl
Saved -> music_kg_triples.tsv  (51,806 rows)
Saved -> music_kg_entities.tsv  (42,032 entities)
Saved -> music_kg_relations.tsv (439 relations)


## 11. PyKEEN quick-check (optional)

If `pykeen` is installed, verify the TSV loads correctly.

In [14]:
try:
    from pykeen.triples import TriplesFactory
    tf = TriplesFactory.from_path("music_kg_triples.tsv",
                                   delimiter="\t")
    print("PyKEEN TriplesFactory loaded successfully:")
    print(f"  num_triples   : {tf.num_triples:,}")
    print(f"  num_entities  : {tf.num_entities:,}")
    print(f"  num_relations : {tf.num_relations:,}")
except ImportError:
    print("PyKEEN not installed — skipping. Install with: pip install pykeen")
except Exception as e:
    print(f"PyKEEN check failed: {e}")

PyKEEN not installed — skipping. Install with: pip install pykeen


## 12. Expansion summary

In [15]:
summary = pd.DataFrame([
    {"Stage": "Step 1–3 (Private KB + alignment)",
     "Triples": "~300", "Note": "Seed KB"},
    {"Stage": "Wave 1 — 1-hop from aligned entities",
     "Triples": wave1_added, "Note": "300 triples/entity × aligned entities"},
    {"Stage": "Wave 2 — predicate-controlled",
     "Triples": wave2_added, "Note": "11 key predicates × 3K–8K rows"},
    {"Stage": "Wave 3 — 2-hop chains",
     "Triples": wave3_added, "Note": "6 join patterns × 8K–10K rows"},
    {"Stage": "Wave 4 — top-up",
     "Triples": topup_added, "Note": "4 broad domain sweeps"},
    {"Stage": "After cleaning & filtering",
     "Triples": n_triples, "Note": "URI-only, connected, deduplicated"},
])
display(summary)

print("\nFinal output files:")
for f in ["music_kg_kge.ttl","music_kg_triples.tsv",
          "music_kg_entities.tsv","music_kg_relations.tsv"]:
    import os
    size = os.path.getsize(f) / 1024 if os.path.exists(f) else 0
    print(f"  {f:<35} {size:>8.1f} KB")

,Stage,Triples,Note
0,Step 1–3 (Private KB + alignment),~300,Seed KB
1,Wave 1 — 1-hop from aligned entities,5856,300 triples/entity × aligned entities
2,Wave 2 — predicate-controlled,51000,11 key predicates × 3K–8K rows
3,Wave 3 — 2-hop chains,49198,6 join patterns × 8K–10K rows
4,Wave 4 — top-up,0,4 broad domain sweeps
5,After cleaning & filtering,51806,"URI-only, connected, deduplicated"



Final output files:
  music_kg_kge.ttl                      4040.5 KB
  music_kg_triples.tsv                  6037.8 KB
  music_kg_entities.tsv                 1928.1 KB
  music_kg_relations.tsv                  19.9 KB


In [1]:
!jupyter nbconvert --to html step4_kb_expansion.ipynb


[NbConvertApp] Converting notebook step4_kb_expansion.ipynb to html
[NbConvertApp] Writing 379278 bytes to step4_kb_expansion.html
